# Oppimisprojekti 3, osa 2: puheentunnistus Whisper-mallilla

Tässä notebookissa käytetään Hugging Facen `transformers`-kirjaston Whisper-malleja suomenkielisen puheen tunnistamiseen. Notebook lukee itse nauhoitetut alle 30 sekunnin äänitteet kansiosta `audio_samples` ja ajaa vertailun usealla mallikoolla.

**Tekoälyn käyttö:** Tekoälyä käytettiin apuna notebookin rakenteen suunnittelussa, koodin korjaamisessa ja raporttipohjan muotoilussa. Ääninäytteet on nauhoitettu itse, Whisper-ajot tehdään paikallisesti, ja lopullinen arviointi kirjoitetaan omien ajotulosten perusteella.


## Mel-spektrogrammi omin sanoin

Whisper ei syötä neuroverkolle raakaa ääniaaltoa. Ääni muunnetaan ensin mel-spektrogrammiksi.

- **x-akseli** kuvaa aikaa: vasemmalta oikealle edetään äänitteen alusta loppuun.
- **y-akseli** kuvaa taajuuksia mel-asteikolla. Mel-asteikko painottaa taajuuksia ihmiskuulon kannalta mielekkäämmin kuin lineaarinen hertsiasteikko.
- **väri** kuvaa voimakkuutta desibeleinä. Kirkkaammat tai voimakkaammat värit tarkoittavat, että kyseisellä ajanhetkellä ja taajuusalueella on enemmän energiaa.

Tämä esitysmuoto sopii neuroverkolle paremmin kuin raaka aalto, koska puheen olennaiset piirteet, kuten vokaalit, konsonantit ja äänenpainot, näkyvät paikallisina kuvioina ajan ja taajuuden tasossa. Transformer saa siis järjestetyn piirrejonon, jota se voi käsitellä samaan tapaan kuin osassa 1 tokenijonoa.

In [ ]:
# Tarvittaessa asenna riippuvuudet esimerkiksi:
# %pip install torch transformers librosa soundfile matplotlib pandas jiwer accelerate

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Laite:", device)

## Ääninäytteet

Notebook lukee äänitiedostot automaattisesti kansiosta `audio_samples`. Tässä projektissa käytössä on useita lyhyitä itse nauhoitettuja näytteitä: selkeä puhe, taustahäly, nopeampi tai epäselvempi puhe, murteellinen/puhekielinen puhe sekä englanninkielisiä vertailunäytteitä.

Kaikki nykyiset `.wav`-tiedostot ovat tarkistuksen perusteella 16 kHz mono -ääntä ja alle 30 sekuntia pitkiä, joten ne sopivat hyvin Whisperin kokeiluun.

Jos tiedät jonkin näytteen tarkan sanamuodon, lisää se `oikea_teksti`-kenttään. Silloin notebook voi laskea myös WER-virhemittarin.

In [ ]:
AUDIO_DIR = Path("audio_samples")
AUDIO_DIR.mkdir(exist_ok=True)

sample_descriptions = {
    "selkea_puhe.wav": {
        "kuvaus": "Selkeä suomenkielinen puhe, rauhallinen lukutapa.",
        "oikea_teksti": "",
    },
    "taustahaly.wav": {
        "kuvaus": "Suomenkielinen puhe taustahälyn kanssa.",
        "oikea_teksti": "",
    },
    "aaninayte_nopea.wav": {
        "kuvaus": "Nopeampi tai hieman epäselvempi suomenkielinen puhe.",
        "oikea_teksti": "",
    },
    "murre_aaninayte.wav": {
        "kuvaus": "Puhekielinen tai murteellinen suomenkielinen näyte.",
        "oikea_teksti": "",
    },
    "murre_aaninayte2.wav": {"kuvaus": "Toinen puhekielinen tai murteellinen suomenkielinen näyte.", "oikea_teksti": ""},
    "english_sample1.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample2.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample3.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample_fast.wav": {
        "kuvaus": "Nopea englanninkielinen vertailunäyte.",
        "oikea_teksti": "",
    },
    "selkea_puhe_1.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 1.", "oikea_teksti": ""},
    "selkea_puhe_2.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 2.", "oikea_teksti": ""},
    "selkea_puhe_3.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 3.", "oikea_teksti": ""},
    "selkea_puhe_4.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 4.", "oikea_teksti": ""},
    "selkea_puhe_5.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 5.", "oikea_teksti": ""},
}

supported = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
audio_files = sorted([p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in supported])

print(f"Löytyi {len(audio_files)} äänitiedostoa kansiosta {AUDIO_DIR.resolve()}")
for p in audio_files:
    print("-", p.name)


In [ ]:
def load_audio(path, target_sr=16000):
    y, sr = librosa.load(path, sr=target_sr, mono=True)
    duration = librosa.get_duration(y=y, sr=sr)
    if duration > 30:
        print(f"Varoitus: {path.name} on {duration:.1f} s. Whisperin peruskokeeseen suositellaan alle 30 s pätkiä.")
    return y, sr, duration

for path in audio_files:
    y, sr, duration = load_audio(path)
    desc = sample_descriptions.get(path.name, {}).get("kuvaus", "Kuvaus puuttuu")
    print(f"{path.name}: {duration:.2f} s, {sr} Hz, {desc}")
    display(Audio(y, rate=sr))

## Mel-spektrogrammin visualisointi

Aja seuraava solu yhdelle tai useammalle näytteelle ja liitä havaintosi raporttiin. Kiinnitä huomiota siihen, missä kohtaa puhe, tauot ja mahdollinen taustahäly näkyvät.

In [ ]:
def plot_mel_spectrogram(path, n_mels=80):
    y, sr, duration = load_audio(path)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=160, n_fft=400)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"Aaltomuoto: {path.name}")

    img = librosa.display.specshow(mel_db, sr=sr, hop_length=160, x_axis="time", y_axis="mel", ax=axes[1])
    axes[1].set_title("Mel-spektrogrammi")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()

if audio_files:
    plot_mel_spectrogram(audio_files[0])
else:
    print("Lisää äänitiedostoja audio_samples-kansioon ja aja solu uudelleen.")

## Whisper-mallien vertailu

Oletuksena vertaillaan malleja `tiny` ja `base`, koska ne latautuvat ja ajavat kohtuullisen nopeasti. Lisää listaan esimerkiksi `small` tai `large-v3`, jos käytössä on tarpeeksi muistia ja aikaa.

In [ ]:
MODEL_SIZES = ["tiny", "base"]  # voit kokeilla myös: "small", "medium", "large-v3"
LANGUAGE = "fi"
TASK = "transcribe"
MAX_SAMPLES = None  # vaihda arvoksi 1, jos haluat ensin nopean testiajon

def model_id(size):
    return "openai/whisper-large-v3" if size == "large-v3" else f"openai/whisper-{size}"


def load_whisper(size):
    name = model_id(size)
    print(f"Ladataan malli {name}...", flush=True)
    processor = WhisperProcessor.from_pretrained(name)
    model = WhisperForConditionalGeneration.from_pretrained(name).to(device)
    model.eval()
    print(f"Malli {name} ladattu.", flush=True)
    return processor, model


def transcribe(path, processor, model, language=LANGUAGE, task=TASK):
    y, sr, duration = load_audio(path)
    inputs = processor(y, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    start = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(input_features, language=language, task=task)
    seconds = time.perf_counter() - start

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return text, duration, seconds


In [ ]:
results = []
files_to_run = audio_files[:MAX_SAMPLES] if MAX_SAMPLES else audio_files

print(f"Ajettavia tiedostoja: {len(files_to_run)}", flush=True)
print(f"Mallit: {', '.join(MODEL_SIZES)}", flush=True)

for size in MODEL_SIZES:
    print()
    print(f"Aloitetaan Whisper {size}...", flush=True)
    processor, whisper_model = load_whisper(size)

    for index, path in enumerate(files_to_run, start=1):
        print(f"  [{index}/{len(files_to_run)}] Tunnistetaan: {path.name}", flush=True)
        language = "en" if path.name.startswith("english") else "fi"
        prediction, duration, seconds = transcribe(path, processor, whisper_model, language=language)
        ref = sample_descriptions.get(path.name, {}).get("oikea_teksti", "")
        results.append({
            "malli": size,
            "tiedosto": path.name,
            "kieli": language,
            "kesto_s": round(duration, 2),
            "aika_s": round(seconds, 2),
            "reaaliaikakerroin": round(seconds / duration, 2) if duration else np.nan,
            "oikea_teksti": ref,
            "tunnistus": prediction,
        })
        print(f"    Valmis {seconds:.2f} sekunnissa", flush=True)

    del whisper_model
    if device == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df.to_csv("osa2_whisper_tulokset_oma_ajo.csv", index=False, encoding="utf-8-sig")
display(results_df)
print("Tulokset tallennettu tiedostoon osa2_whisper_tulokset_oma_ajo.csv", flush=True)


In [ ]:
# Valinnainen WER-laskenta.
# Tässä ajossa oikeat sanasta sanaan -referenssit eivät ole tiedossa,
# joten WER jätetään pois. Jos kirjoitat tarkat puhutut lauseet
# sample_descriptions-sanastoon, voit ajaa tämän solun.

try:
    from jiwer import wer
    if not results_df.empty and results_df["oikea_teksti"].fillna("").str.len().gt(0).any():
        results_df["WER"] = results_df.apply(
            lambda r: wer(r["oikea_teksti"], r["tunnistus"]) if r["oikea_teksti"] else np.nan,
            axis=1,
        )
        display(results_df[["malli", "tiedosto", "kieli", "aika_s", "reaaliaikakerroin", "WER", "tunnistus"]])
    else:
        print("WER-laskentaa ei ajettu, koska tarkkoja referenssitekstejä ei ole täytetty.")
except Exception as exc:
    print("WER-laskentaa ei ajettu:", exc)


## Raportointitaulukko ja tulosten analyysi

Tämä osio täydennetään oman ajon jälkeen. Älä käytä valmiiksi kirjoitettuja esimerkkituloksia, vaan aja Whisper-vertailusolu ja muodosta taulukot `results_df`-muuttujasta.

Jos et ole kirjoittanut tarkkoja litteraatteja `oikea_teksti`-kenttiin, älä raportoi WER-lukua. Arvioi silloin laatua laadullisesti: mitkä sanat tunnistuivat oikein, missä tuli virheitä, ja miten virheet erosivat mallien välillä.


In [ ]:
if "results_df" not in globals() or results_df.empty:
    print("Aja ensin Whisper-vertailusolu, jotta results_df muodostuu.")
else:
    comparison = (
        results_df
        .pivot_table(
            index=["tiedosto", "kieli", "kesto_s"],
            columns="malli",
            values=["aika_s", "reaaliaikakerroin", "tunnistus"],
            aggfunc="first",
        )
        .sort_index()
    )

    averages = (
        results_df
        .groupby("malli", as_index=False)
        .agg(
            naytteita=("tiedosto", "count"),
            keskim_aika_s=("aika_s", "mean"),
            keskim_reaaliaikakerroin=("reaaliaikakerroin", "mean"),
        )
        .round(2)
    )

    display(comparison)
    display(averages)

    print("Kirjoita analyysiin oman ajosi perusteella esimerkiksi:")
    print("- kumpi malli oli nopeampi ja kuinka suuri ero oli")
    print("- missä näytteissä tunnistus onnistui parhaiten")
    print("- missä näytteissä tuli selviä virheitä")
    print("- vaikuttiko taustahäly, nopea puhe, murre tai englannin kieli tuloksiin")


### Analyysi oman ajon jälkeen

Täydennä tähän lopulliset havainnot vasta oman ajon jälkeen.

- **Nopeus:** Kirjaa, mikä malli oli nopeampi ja millaiset keskimääräiset ajoajat sait.
- **Tarkkuus:** Kuvaa esimerkkien avulla, kumman mallin teksti vastasi paremmin puhetta.
- **Vaikeat näytteet:** Arvioi, miten taustahäly, nopea puhe ja murteellisuus vaikuttivat tunnistukseen.
- **Englanti vs. suomi:** Vertaa, toimiko englanninkielinen puhe paremmin, huonommin vai suunnilleen samalla tasolla kuin suomenkielinen puhe.

### Yhteys osaan 1

Osassa 1 transformer käsitteli tekstin tokenijonoa ja ennusti seuraavaa tokenia. Whisperissä syöte ei ole tekstinä, vaan ääni muunnetaan ensin mel-spektrogrammiksi eli aika-taajuuspiirteiden jonoksi. Molemmissa tapauksissa transformer hyödyntää attention-mekanismia järjestetyn syötteen riippuvuuksien mallintamiseen. Tämä on transformerien vahvuus: sama perusidea sopii sekä tekstin että puheen kaltaisiin sekventiaalisiin aineistoihin.
